In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_teddynote.graphs import visualize_graph
from langchain_teddynote import logging
from dotenv import load_dotenv
import os

# API KEY 정보로드
load_dotenv()

# 프로젝트 이름을 입력합니다.
logging.langsmith(os.environ["STUDENT_NAME"] + "-" + "Simple_Agent_Chat")



In [ ]:
###### STEP 1. 상태(State) 정의 ######
class State(TypedDict):
    # 메시지 정의(list type 이며 add_messages 함수를 사용하여 메시지를 추가)
    messages: Annotated[list, add_messages]


###### STEP 2. 노드(Node) 정의 ######
# LLM 정의
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


# 챗봇 함수 정의
def chatbot(state: State):
    # 메시지 호출 및 반환
    return {"messages": [llm.invoke(state["messages"])]}



In [ ]:
###### STEP 3. 그래프(Graph) 정의, 노드 추가 ######
# 그래프 생성
graph_builder = StateGraph(State)

# 노드 이름, 함수 혹은 callable 객체를 인자로 받아 노드를 추가
graph_builder.add_node("chatbot", chatbot)

###### STEP 4. 그래프 엣지(Edge) 추가 ######
# 시작 노드에서 챗봇 노드로의 엣지 추가
graph_builder.add_edge(START, "chatbot")

# 그래프에 엣지 추가
graph_builder.add_edge("chatbot", END)



In [ ]:
###### STEP 5. 그래프 컴파일(compile) ######
# 그래프 컴파일
graph = graph_builder.compile()

###### STEP 6. 그래프 시각화 ######
# 그래프 시각화
visualize_graph(graph)



In [ ]:
###### STEP 7. 그래프 실행 ######
SYSTEM_PROMPT = """
    너는 주식 투자 전문가이다.
    문의한 종목에대해 매수/매도/홀딩 의견을 주고, 관련 근거도 제시해야 한다.
"""

question = "삼성전자 주식을 매수 해도 되나요?"

# 그래프 이벤트 스트리밍
for event in graph.stream(
    {"messages": [("system", SYSTEM_PROMPT), ("user", question)]}
):
    # 이벤트 값 출력
    for value in event.values():
        print(value["messages"][-1].content)

In [ ]:
###### STEP 7. 그래프 실행 ######
SYSTEM_PROMPT = """
    너는 주식 투자 전문가이다.
    문의한 종목에대해 매수/매도/홀딩 의견을 주고, 관련 근거도 제시해야 한다.
    
    # 답변은 아래예시로 부탁해.
    1. 의견: 매수
    2. 근거: 
        - 근거1
        - 근거2
        - 근거3
    3. 종합:
"""

question = "SK하이닉스 주식을 매수 해도 되나요?"

# 그래프 이벤트 스트리밍
for event in graph.stream(
    {"messages": [("system", SYSTEM_PROMPT), ("user", question)]}
):
    # 이벤트 값 출력
    for value in event.values():
        print(value["messages"][-1].content)